In [3]:
!pip install -q langchain_google_genai langchain langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 79.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [4]:
import os
from google.colab import userdata

os.environ['LANGSMITH_API_KEY'] = userdata.get('LANGSMITH_API_KEY')
os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')

In [5]:
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://eu.api.smith.langchain.com'
os.environ['LANGCHAIN_PROJECT'] = 'chatbot_w_langchain_memory'

In [6]:
import warnings
warnings.filterwarnings('ignore')

In [7]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

In [9]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model = "gemini-3.1-flash-lite", convert_system_message_to_human = True)

In [12]:
while True:
  message = input("write ur query")
  if message == "bye":
    print("Have a great day!")
    break
  else:
    if message is not None:
      print(parser.invoke(model.invoke([HumanMessage(content=message)])))
    else:
      print("No message to process")

write ur queryhi
Hello! How can I help you today?
write ur queryim pragi
Hello, Pragi! It's nice to meet you. How are you doing today? Is there anything I can help you with?
write ur querywhats my name?
I don’t know your name! As an AI, I don’t have access to your personal information, your camera, or your identity unless you choose to share that information with me in our conversation. 

If you'd like, you can tell me your name, and I'll remember it for the duration of this chat!
write ur querybye
Have a great day!


In [18]:
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

In [19]:
store = {}

In [21]:
def get_session_history(session_id : str) -> BaseChatMessageHistory:
  if session_id not in store:
    store[session_id] = InMemoryChatMessageHistory()
  return store[session_id]

In [23]:
model_with_memory = RunnableWithMessageHistory(model,get_session_history)

In [25]:
model_with_memory.invoke(
    {"input": "Hello, my name is Alex"},
    config={"configurable": {"session_id": "user_123"}}
).content

[{'type': 'text',
  'text': "Hello again, Alex! It's nice to see you. How can I help you today?",
  'extras': {'signature': 'EjQKMgEMOdbHrbP99I4jriMLGF2CtHu3gpM2OtVg0NGsihZ+0ulWy0TEuYtupYuiclesatkF'}}]

In [26]:
print(store)

{'user_123': InMemoryChatMessageHistory(messages=[HumanMessage(content='Hello, my name is Alex', additional_kwargs={}, response_metadata={}), AIMessage(content=[{'type': 'text', 'text': "Hello, Alex! It's nice to meet you. How are you doing today? Is there anything I can help you with?", 'extras': {'signature': 'EjQKMgEMOdbHghWTZdxOLinEMRdEufFB08zZUpIEh/0mRTEuiHcdRAO7auFrlqD9/rmtdgZ3'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019eccb1-78d3-7272-b32d-b4c07dfa523c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 7, 'output_tokens': 27, 'total_tokens': 34, 'input_token_details': {'cache_read': 0}}), HumanMessage(content='Hello, my name is Alex', additional_kwargs={}, response_metadata={}), AIMessage(content=[{'type': 'text', 'text': "Hello again, Alex! It's nice to see you. How can I help you today?", 'extras': {'signature': 'EjQK

In [27]:
model_with_memory.invoke(
    {"input": "whats my name"},
    config={"configurable": {"session_id": "user_1234"}}
).content

[{'type': 'text',
  'text': "I don’t know your name. As an AI, I don’t have access to your personal information, identity, or private data unless you’ve shared it with me during our conversation. \n\nIf you'd like, you can tell me your name, and I'll be happy to remember it while we're chatting!",
  'extras': {'signature': 'EjQKMgEMOdbHSmEDfTa0WYYCgx+H1s1g95vw7+0JRHMY1veQ4NQsBRmX1uWciLm7dyHSw+Zo'}}]

In [28]:
print(store)

{'user_123': InMemoryChatMessageHistory(messages=[HumanMessage(content='Hello, my name is Alex', additional_kwargs={}, response_metadata={}), AIMessage(content=[{'type': 'text', 'text': "Hello, Alex! It's nice to meet you. How are you doing today? Is there anything I can help you with?", 'extras': {'signature': 'EjQKMgEMOdbHghWTZdxOLinEMRdEufFB08zZUpIEh/0mRTEuiHcdRAO7auFrlqD9/rmtdgZ3'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019eccb1-78d3-7272-b32d-b4c07dfa523c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 7, 'output_tokens': 27, 'total_tokens': 34, 'input_token_details': {'cache_read': 0}}), HumanMessage(content='Hello, my name is Alex', additional_kwargs={}, response_metadata={}), AIMessage(content=[{'type': 'text', 'text': "Hello again, Alex! It's nice to see you. How can I help you today?", 'extras': {'signature': 'EjQK

In [29]:
model_with_memory.invoke(
    {"input": "whats my name"},
    config={"configurable": {"session_id": "user_123"}}
).content

[{'type': 'text',
  'text': 'Your name is Alex!',
  'extras': {'signature': 'EjQKMgEMOdbHtaUiLW4iXg9Ol2gRyICIzYfZ+r+fJb39F57oQIct/MVbZpBZLlTmUAe3S6v5'}}]

In [30]:
print(store)

{'user_123': InMemoryChatMessageHistory(messages=[HumanMessage(content='Hello, my name is Alex', additional_kwargs={}, response_metadata={}), AIMessage(content=[{'type': 'text', 'text': "Hello, Alex! It's nice to meet you. How are you doing today? Is there anything I can help you with?", 'extras': {'signature': 'EjQKMgEMOdbHghWTZdxOLinEMRdEufFB08zZUpIEh/0mRTEuiHcdRAO7auFrlqD9/rmtdgZ3'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019eccb1-78d3-7272-b32d-b4c07dfa523c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 7, 'output_tokens': 27, 'total_tokens': 34, 'input_token_details': {'cache_read': 0}}), HumanMessage(content='Hello, my name is Alex', additional_kwargs={}, response_metadata={}), AIMessage(content=[{'type': 'text', 'text': "Hello again, Alex! It's nice to see you. How can I help you today?", 'extras': {'signature': 'EjQK

In [31]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage

# 1. PROMPT
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="messages")  # history slots in here
])

# 2. CHAIN
chain = prompt | model | StrOutputParser()

# 4. CHAIN WITH MEMORY
chain_with_memory = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

# 5. INVOKE
config = {"configurable": {"session_id": "user_123"}}

response = chain_with_memory.invoke(
    {"messages": [HumanMessage(content="What's my name?")]},
    config=config
)
print(response)

Your name is Alex!


In [33]:
print(store)

{'user_123': InMemoryChatMessageHistory(messages=[HumanMessage(content='Hello, my name is Alex', additional_kwargs={}, response_metadata={}), AIMessage(content=[{'type': 'text', 'text': "Hello, Alex! It's nice to meet you. How are you doing today? Is there anything I can help you with?", 'extras': {'signature': 'EjQKMgEMOdbHghWTZdxOLinEMRdEufFB08zZUpIEh/0mRTEuiHcdRAO7auFrlqD9/rmtdgZ3'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019eccb1-78d3-7272-b32d-b4c07dfa523c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 7, 'output_tokens': 27, 'total_tokens': 34, 'input_token_details': {'cache_read': 0}}), HumanMessage(content='Hello, my name is Alex', additional_kwargs={}, response_metadata={}), AIMessage(content=[{'type': 'text', 'text': "Hello again, Alex! It's nice to see you. How can I help you today?", 'extras': {'signature': 'EjQK

In [35]:
from langchain_core.messages import trim_messages, HumanMessage, AIMessage, SystemMessage

# 1. DEFINE THE TRIMMER
trimmer = trim_messages(
    max_tokens=40,
    strategy="last",              # keep the most RECENT messages
    token_counter=model,          # uses the model to count tokens
    include_system=True,          # always keep the system message
    allow_partial=False,          # don't cut a message halfway
    start_on="human"              # trimmed result must start with a human message
)

# 2. YOUR MESSAGES (~50 tokens)
messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="My name is Alex."),
    AIMessage(content="Nice to meet you Alex!"),
    HumanMessage(content="I love cricket."),
    AIMessage(content="Cricket is a great sport!"),
    HumanMessage(content="What's my name?"),
]

# 3. CHECK TOKEN COUNT BEFORE TRIMMING
print(model.get_num_tokens_from_messages(messages))  # returns ~50

# 4. TRIM
trimmed_messages = trimmer.invoke(messages)

# 5. CHECK AFTER
print(model.get_num_tokens_from_messages(trimmed_messages))  # now ≤ 40

45
30


In [36]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

chain = (
    RunnablePassthrough.assign(messages=itemgetter("messages") | trimmer)
    | prompt
    | model
    | StrOutputParser()
)

response = chain.invoke(
    {"messages": messages},  # your full message list
    config={"configurable": {"session_id": "user_123"}}
)

print(response)

I don't know your name yet! Since this is our first conversation, you haven't told me what it is. 

What should I call you?
